In [ ]:
from suite2p.extraction import masks, extract
from suite2p.detection import stats
from suite2p.io import binary

if not with_func:
    print("Skipping this code block since with_func = False (classification only, no trace extraction)")
else:
    cell_masks = []
    for i in range(1, numROIs+1):
        idx = np.where(res[0].flatten()==i)
        cell_masks.append((idx[0], np.ones(idx[0].shape)))
    
    
    if isinstance(ops['reg_file'], str):
        # Only one binary file
        reader = binary.BinaryFile(
            filename=ops['reg_file'],
            Lx=ops['Lx'],
            Ly=ops['Ly'],
            n_frames=ops['frames_per_file'][0]
        )
        Y = reader.data # shape: (n_frames, Ly, Lx)
    else:
        # Multiple binary files
        all_data = []
        for fname, n_frames in zip(ops['reg_file'], ops['frames_per_file']):
            reader = BinaryFile(
                filename=fname,
                Lx=ops['Lx'],
                Ly=ops['Ly'],
                n_frames=n_frames
            )
            all_data.append(reader.data)
    
        Y = np.concatenate(all_data, axis=0)  # shape: (total_frames, Ly, Lx)

    # identify neuropil masks for cellpose ROIs (look for built in suite2p function that does this; then change neuropil_masks from None)
    
    # extract ROI traces using suite2p
    F_CP, Fneu_CP = extract.extract_traces(f_in=Y, cell_masks=cell_masks, neuropil_masks=None) # Fneu_CP should be empty for now
    
    # Save results
    path = os.path.join(cellpose_dir, "F_cp.npy")
    if os.path.exists(path):
        print("Did not save to avoid overwriting previously saved F_cp.npy")
    else:
        np.save(path, F_CP)

In [ ]:
# to avoid double-counting, record amount of overlap between cellpose and suite2p ROIs

# cellpose ROIs are in res[0]
# suite2p ROIs are in stats
if not with_func:
    print("Skipping this code block since with_func = False (classification only, no trace extraction)")
else:
    overlap = np.full((np.max(res[0]), 2), np.nan) # first column = max overlap fraction, second column = which suite2p ROI had max overlap
    stat = np.load(os.path.join(suite2p_path, "stat.npy"), allow_pickle=True)
    iscell = np.load(os.path.join(suite2p_path, "iscell.npy"), allow_pickle=True)
    for i in range(len(overlap)):
        shared = []
        s2p_idx = []
        ypix, xpix = np.where(res[0]==i)
        A = np.column_stack((ypix.ravel(), xpix.ravel()))
        A_view = A.view([('', A.dtype)]*2).ravel()
        for j in range(len(stat)):
            #if iscell[j, 0]==1:
            ypix_s2p = stat[j]["ypix"]
            xpix_s2p = stat[j]["xpix"]
            
            B = np.column_stack((ypix_s2p.ravel(), xpix_s2p.ravel()))
            B_view = B.view([('', B.dtype)]*2).ravel()
            
            shared.append(np.isin(A_view, B_view).sum())
            s2p_idx.append(j)
        
        overlap[i, :] = np.array([np.max(shared)/len(ypix), s2p_idx[np.argmax(shared)]])

    # Save results
    path = os.path.join(cellpose_dir, "overlap.npy")
    if os.path.exists(path):
        print("Did not save to avoid overwriting previously saved overlap.npy")
    else:
        np.save(path, overlap)

# have one stat file that contains both lists of ROIs, with overlapping ROIs excluded (>75%)
# isredcell.npy
# loadable in the suite2p GUI
# PROBLEM: stat file doesn't have all the right stuff
# email/github suite2p
# put isredcell

In [ ]:
from suite2p.detection import stats
dst = os.path.join(os.path.join(os.path.join(func_dir, "interleaved"), "combined"), "plane0")
'''
# copy entire suite2p output to new directory
import shutil

src = suite2p_path
dst = os.path.join(os.path.join(func_dir, "interleaved"), "combined")

os.mkdir(dst)

shutil.copytree(src, dst, dirs_exist_ok=True)
print("finished copying files")
'''
# combine old and new stat, F, Fneu, iscell, redcell
stat_new_temp = np.array(labels_to_stat(res[0])) # change back to res[0]
stat_new = stats.roi_stats(stat_new_temp, res[0].shape[0], res[0].shape[1], aspect=None, diameter=None, max_overlap=None, do_crop=True)

# FIX THIS
for i in range(len(stat_new)):
    stat_new[i]['skew'] = 0
    stat_new[i]['std'] = 1 
    stat_new[i]['neuropil_mask'] = np.array([])

stat_old = np.load(os.path.join(suite2p_path, "stat.npy"), allow_pickle=True)

stat = np.concatenate((stat_old, stat_new))

F_new = F_CP
Fneu_new = np.zeros(F_CP.shape)

F_old = np.load(os.path.join(suite2p_path, "F.npy"), allow_pickle=True)
Fneu_old = np.load(os.path.join(suite2p_path, "Fneu.npy"), allow_pickle=True)

F = np.concatenate((F_old, F_new), axis=0)
Fneu = np.concatenate((Fneu_old, Fneu_new), axis=0)

spks_old = np.load(os.path.join(suite2p_path, "spks.npy"), allow_pickle=True)
spks_new = np.zeros((F_CP.shape[0], spks_old.shape[1]))
spks = np.concatenate((spks_old, spks_new), axis=0)

iscell_old = np.load(os.path.join(suite2p_path, "iscell.npy"), allow_pickle=True)
iscell_new = np.ones((F_CP.shape[0], 2))

iscell = np.concatenate((iscell_old, iscell_new), axis=0)

redcell_old = np.load(os.path.join(suite2p_path, "redcell.npy"), allow_pickle=True)
redcell_new = np.ones(iscell_new.shape) # all of the "new" cells are red per cellpose

redcell = np.concatenate((redcell_old, redcell_new), axis=0)

# change ops file paths
ops_new = np.load(os.path.join(suite2p_path, "ops.npy"), allow_pickle=True)
ops_new_path = os.path.join(dst, "ops.npy")
ops_new[()]['ops_path'] = ops_new_path
ops_new[()]['save_path'] = dst
ops_new[()]['reg_file'] = os.path.join(dst, "data.bin")
ops_new[()]['reg_file_chan2'] = os.path.join(dst, "data_chan2.bin")

np.save(os.path.join(dst, "ops.npy"), ops_new)
np.save(os.path.join(dst, "F.npy"), F)
np.save(os.path.join(dst, "Fneu.npy"), Fneu)
np.save(os.path.join(dst, "spks.npy"), spks)
np.save(os.path.join(dst, "iscell.npy"), iscell)
np.save(os.path.join(dst, "redcell.npy"), redcell)
np.save(os.path.join(dst, "stat.npy"), stat)

# closer...will actually open up in suite2p...but won't show ROIs
# now not working at all
# probably need to add more arguments to stat_new to make it exactly the same as stat_old

# now try opening in suite2p GUI
'''
path = os.path.join(traces_dir, "stat.npy")
if os.path.exists(path):
    print("Did not save to avoid overwriting previously saved stat.npy")
else:
    np.save(path, np.array(stat, dtype=object))

# also need to copy ops to this directory
suite2p_path = os.path.join(os.path.join(os.path.join(func_dir, "interleaved"), "suite2p"), "plane0")
ops = np.load(os.path.join(suite2p_path, "ops.npy"), allow_pickle=True)[()]
np.save(os.path.join(traces_dir, "ops.npy"), ops)

# also need to add a dummy iscell
iscell = np.ones((len(stat), 2), dtype=np.float32)
np.save(os.path.join(traces_dir, "iscell.npy"), iscell)

# still getting "incorrect files. Choose another?" error in suite2p
'''